# Projected finite-difference gradient descent: 100 spectral factors

**Same seeded mask → restore brightness if needed → measure finite-step slopes →
project and backtrack → validate → compare measured performance.**

Based on [the spectral-mask LLMBO notebook](../LLM-assisted%20Bayesian%20optimization/dnl_llm_bo_spectral_mask.ipynb), which is
preserved. Select **GNLSE (Python)**. Function details are in
[the descent helper](../../../notebook_helpers/llm_bo/spectral_mask_gd.py) and
[the reporting helper](../../../notebook_helpers/llm_bo/spectral_mask_gd_report.py).

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "notebook_helpers" / "run_library.py").is_file())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "notebook_helpers"))
from llm_bo.spectral_mask_gd import run_experiment
from llm_bo.spectral_mask_gd_report import (
    show_descent_route, show_results, show_comparison, show_pulse_comparison,
)
get_ipython().run_line_magic("matplotlib", "inline")

## 1. Same physics, mask and starting point

The unmasked input is a **100 kW, 200 fs Gaussian**, with equal amplitudes and zero
relative phase in LP01, LP11-cos and LP11-sin. The original silica step-index fiber
is retained: 2.86 m length, 52.5 μm core radius, NA 0.1, and 1379.2 nm carrier.
The existing solver supplies the same dispersion, nonlinear response and CP overlap tensor.

The **100 equal-width angular-frequency bands** span the centred interval containing
99.9% of the original pulse's spectral energy. Their physical edges stay fixed.
One field-amplitude mask is shared by all three modes:

\[
\widetilde A_{\mathrm{shaped}}(\omega)=s_j\widetilde A(\omega),\qquad
s_j\in\{0,0.01,\ldots,1\}.
\]

Frequencies outside the controlled bandwidth are unchanged. There is **no
renormalization**: a factor of 0.5 retains 0.25 of that band's power. The transform
convention remains `fftshift(ifft(A))`, with inverse `fft(ifftshift(spectrum))`.

Minimize the coherent **all-mode D_NL** between matched linear (`n2=0`) and nonlinear
outputs, including all three modes and without temporal alignment. The output-photon
floor is **50% of the unmasked 1 mm baseline**. Each objective evaluation is a matched
pair of propagations. The baseline and initial mask are measured again for this run.

The search uses **Nt=16384, T=64.87042 ps, dz=5 mm**, with four samples per band.
The seed below is identical to the saved LLMBO run. `LOAD_RUN` loads the measured
run without new simulations; set it to `None` to start a new bounded campaign.

In [ ]:
SEED = 20261113
MAX_ITERATIONS = 2
MAX_EVALUATIONS = 450      # Objective pairs; includes restoration, probes and checks.
MAX_MINUTES = 45
VALIDATION_MINUTES = 6     # Reserved inside MAX_MINUTES, not added to it.
LOAD_RUN = None
LLM_REFERENCE = None

## 2. Restore feasibility, then descend

The short excerpts show the logic; the run cell executes the full routine.

### 2.1 Start from the same seeded random mask

```python
factors = np.random.default_rng(SEED).integers(0, 101, 100) / 100
```

If the measured photon ratio is below 0.50, move toward the unmasked launch using
`t = 0.1, 0.2, ..., 1.0`, projecting each proposal onto the allowed grid:

```python
restored = project(factors + t * (1 - factors))
```

Stop at the **first measured feasible point**. This restoration may increase D_NL;
its purpose is to satisfy brightness, and it is reported separately from descent.
The original random mask, every restoration proposal, and their results are saved.

### 2.2 Measure finite-step slopes

For each of the 100 factors, perturb by **0.01**. Use central differences when
both neighbours are allowed; use forward differences at 0 and backward differences at 1.

```python
h = 0.01
slope = (D_plus - D_minus) / (2 * h)  # Interior factor.
slope_at_zero = (D_plus - D_current) / h
slope_at_one = (D_current - D_minus) / h
```

These are **finite-step slopes**, not exact derivatives. No differentiation passes
through rounding. Infeasible probes may inform a slope, but cannot be accepted as
optimization steps. The current objective and repeated probes are cached.
A full interior gradient costs **200 objective pairs (400 GNLSE propagations)**;
the saved pilot has an explicit two-iteration cap.

### 2.3 Project, backtrack and accept

Scale the initial step so the largest proposed factor change is 0.10. Clip to
[0,1], round to the nearest 0.01 (ties upward), then measure the projected mask.

```python
alpha = 0.10 / max(abs(slopes))
proposal = project(factors - alpha * slopes)
accept = result["feasible"] and result["D_NL"] < current["D_NL"] - 1e-10
```

If rejected, halve `alpha`, for at most **12 proposals**. Only a feasible objective
improvement becomes the next iterate. An unchanged projected mask ends backtracking.
The selected result is the best unmasked baseline, restored start or accepted iterate;
individual probes and rejected proposals do not replace the iterate.

Stop after the explicit iteration, evaluation or wall-time limit, or if backtracking
finds no accepted step. Cache hits cost no additional pair. Three final-check pairs
and six minutes are reserved inside the total budget. Unfinished checks remain
`not_run`; a small pilot does not establish convergence.

The controller records active and calendar clocks separately and checks the calendar
deadline during a pending simulation. If the computer pauses, it stops work when it
next runs and observes an expired deadline. The saved pilot predates this clock fix;
its timing difference is reported in section 4.

Every pair saves its request, response, fields and duration. `events.jsonl` records
gradient probes, restoration, accepted steps, rejected proposals, cache hits and
cumulative elapsed time; `gradient_*.json` stores all slope numerators and evidence IDs.

In [ ]:
# File the saved campaign under its launch controls and search method.
from notebook_helpers.run_library import file_saved_run

run = run_experiment(
    seed=SEED, max_iterations=MAX_ITERATIONS, max_evaluations=MAX_EVALUATIONS,
    max_minutes=MAX_MINUTES, validation_minutes=VALIDATION_MINUTES,
    load_directory=LOAD_RUN,
)
file_saved_run(run.directory, "Spectral amplitude mask", "Gradient descent")

## 3. Descent route, selected mask and numerical checks

First show the saved stopping reason and the **accepted GD route**. Step 0 is the
first feasible mask after restoration. Rejected proposals are measured tests; they
do not advance the accepted iterate. The history plot connects only accepted GD iterates;
initialization, restoration and rejected proposals appear as separate markers.
Backtracking starts each proposal from the current accepted mask, not the previous
rejected proposal.

The remaining plots retain the input-energy and brightness histories, 100 factors,
input/output spectra, pulse centre and full-window temporal tails. All gradient
probes remain in the full saved evaluation ledger.

Recheck the unmasked baseline and selected mask at **dz=1 mm** on the search grid,
then at **Nt=65536, T=129.74084 ps**, which doubles the window and halves dt.
Use the same physical band edges and factors. The absolute D_NL tolerance is
**0.0002** for propagation-step and time/frequency-grid comparisons. The refined
brightness check uses the unmasked 1 mm reference on the same refined grid, as in LLMBO.

Failed or missing checks leave the result **preliminary**. Raw scores remain visible;
attenuation and pulse broadening are allowed controls and do not by themselves
establish improved performance for an application.

In [ ]:
show_descent_route(run)
show_results(run)

## Compare with your spectral-mask LLM run (optional)

Set `LLM_REFERENCE` above to the run directory printed by the spectral-mask LLM notebook.

In [ ]:
if LLM_REFERENCE is not None:
    comparison = show_comparison(run, LLM_REFERENCE)

### LLAMBO beside GD: input, linear output and nonlinear output

**LLAMBO on the left; GD on the right.** Each panel shows the selected **masked input**,
its **matched linear output** and its **nonlinear output**, using the saved 1 mm
propagation checks on the same grid. Compare the pulse centre, wavelength spectrum
and full time window on shared absolute scales, summing all three modes without
renormalization or time alignment. These are the selected shaped pulses; the common
100 kW Gaussian is the launch **before** shaping.

In [ ]:
if LLM_REFERENCE is not None:
    show_pulse_comparison(run, LLM_REFERENCE)